In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Путь к данным (укажите свой путь после скачивания с Kaggle)
# Структура обычно такая: /device_name/benign_traffic.csv, /device_name/mirai_attacks/udp.csv и т.д.
path = '/home/lolkek3310/python/Подготовка Гомель/боты/N-BaIoT' 

def load_data(device_id='1'):
    """
    Загружает данные для конкретного устройства по его ID.
    device_id: строка ('1', '2' ... '9') согласно префиксам ваших файлов.
    """
    all_dfs = []
    
    # Получаем список всех файлов в директории
    files = os.listdir(path)
    
    # Фильтруем только те файлы, которые относятся к выбранному устройству
    # Например, если device_id = '1', выберем '1.benign.csv', '1.mirai.udp.csv' и т.д.
    device_files = [f for f in files if f.startswith(f"{device_id}.")]
    
    if not device_files:
        print(f"Файлы для устройства {device_id} не найдены!")
        return None

    for file in device_files:
        file_path = os.path.join(path, file)
        
        # Загружаем CSV
        df_temp = pd.read_csv(file_path)
        
        # Если в названии файла есть слово 'benign' — это нормальный трафик (0)
        # В противном случае (mirai или gafgyt) — это атака (1)
        if 'benign' in file:
            df_temp['target'] = 0
            print(f"Загружен чистый трафик: {file} ({len(df_temp)} строк)")
        else:
            df_temp['target'] = 1
            # Печатаем для контроля, что именно мы считаем атакой
            print(f"Загружена атака: {file} ({len(df_temp)} строк)")
            
        all_dfs.append(df_temp)
        
    # Объединяем все найденные файлы устройства в одну таблицу
    full_df = pd.concat(all_dfs, axis=0, ignore_index=True)
    return full_df

# Загружаем данные для первого устройства (префикс 1.)
df = load_data(device_id='1')

if df is not None:
    print(f"\nИтоговый размер собранного датасета: {df.shape}")
    print(f"Распределение классов:\n{df['target'].value_counts()}")

Загружен чистый трафик: 1.benign.csv (49548 строк)
Загружена атака: 1.gafgyt.combo.csv (59718 строк)
Загружена атака: 1.gafgyt.junk.csv (29068 строк)
Загружена атака: 1.gafgyt.scan.csv (29849 строк)
Загружена атака: 1.gafgyt.tcp.csv (92141 строк)
Загружена атака: 1.gafgyt.udp.csv (105874 строк)
Загружена атака: 1.mirai.ack.csv (102195 строк)
Загружена атака: 1.mirai.scan.csv (107685 строк)
Загружена атака: 1.mirai.syn.csv (122573 строк)


In [ ]:
# Проверка пропусков
print(df.isnull().sum().sum())

# Баланс классов
sns.countplot(x='target', data=df)
plt.title('Распределение классов (0 - Норма, 1 - Атака)')
plt.show()

# Корреляция признаков (выберем топ самых коррелирующих с таргетом)
correlations = df.corr()['target'].sort_values(ascending=False)
print("Топ признаков:\n", correlations.head(10))

In [ ]:
# Разделяем на признаки и целевую переменную
X = df.drop('target', axis=1)
y = df['target']

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Масштабирование признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Обучающая выборка: {X_train.shape}")

In [ ]:
# Используем Random Forest, так как он устойчив к выбросам и не требует сложной настройки
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

print("Начало обучения...")
model.fit(X_train, y_train)
print("Модель обучена!")

In [ ]:
y_pred = model.predict(X_test)

# Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Предсказано')
plt.ylabel('Реальность')
plt.show()

# Подробный отчет
print(classification_report(y_test, y_pred))

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[-10:]  # Топ-10 признаков

plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.title('Важность признаков')
plt.show()